# Week 14 — Governance

**Focus:** How does a model result become a fair, explainable, and monitored decision?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week14/week14_demos.ipynb)

## What this notebook is about

A financial model can be accurate on average and still create unacceptable errors for particular people or groups. This notebook treats fairness, explanation, monitoring, and documentation as parts of the model system rather than as decorations added at the end.

Using a synthetic loan book, you will expose a proxy, measure selection-rate disparities, trace a fairness-policy frontier, generate reason codes, compare model-agnostic importance, monitor population and performance drift, and assemble a model-card record.

By the end, you should be able to:

- distinguish a protected attribute from a proxy that carries related information;
- compute and interpret an impact ratio without treating it as a complete legal test;
- describe the tradeoffs created by a policy intervention;
- separate local reasons from global feature importance; and
- connect monitoring thresholds with ownership and an action plan.

The loan data are synthetic and the diagnostics are educational. They illustrate questions a governed system must answer; they are not legal advice or a compliance certification.

### Course context

For a concise review and the optional textbook route, see [Week 14 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week14/week14_summary.md) and [Week 14 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week14/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — Build a synthetic loan book and expose the proxy — <a href="#Demo-1-%E2%80%94-Build-a-synthetic-loan-book-and-expose-the-proxy">Jupyter</a> · <a href="#scrollTo=md-c1b64f867624">Colab</a></li>
<li>Demo 2 — 4/5ths rule and impact ratio — <a href="#Demo-2-%E2%80%94-4%2F5ths-rule-and-impact-ratio">Jupyter</a> · <a href="#scrollTo=md-3a628672b3f8">Colab</a></li>
<li>Demo 3 — Fairness-policy tradeoff map — <a href="#Demo-3-%E2%80%94-Fairness-policy-tradeoff-map">Jupyter</a> · <a href="#scrollTo=md-ebf3c756ef37">Colab</a></li>
<li>Demo 4 — Candidate adverse contributions from the logistic model — <a href="#Demo-4-%E2%80%94-Candidate-adverse-contributions-from-the-logistic-model">Jupyter</a> · <a href="#scrollTo=md-40d9f4fd7745">Colab</a></li>
<li>Demo 5 — Permutation importance (model-agnostic, pure NumPy) — <a href="#Demo-5-%E2%80%94-Permutation-importance-(model-agnostic,-pure-NumPy)">Jupyter</a> · <a href="#scrollTo=md-c2fdfa63a924">Colab</a></li>
<li>Demo 6 — Drift monitoring (PSI + delayed outcomes) over 24 months — <a href="#Demo-6-%E2%80%94-Drift-monitoring-(PSI-%2B-delayed-outcomes)-over-24-months">Jupyter</a> · <a href="#scrollTo=md-c458e2b836e4">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 7 — Model card stub (what you&#x27;d produce alongside the model) — <a href="#Demo-7-%E2%80%94-Model-card-stub-(what-you%27d-produce-alongside-the-model)">Jupyter</a> · <a href="#scrollTo=md-acb874bba749">Colab</a></li>
<li>Demo 8 — Repair a result slide without changing the result — <a href="#Demo-8-%E2%80%94-Repair-a-result-slide-without-changing-the-result">Jupyter</a> · <a href="#scrollTo=md-960e73386af5">Colab</a></li>
</ul>

## Read the code: audit the decision people experience

Begin with the action and the population: who applies, who is approved, and whose repayment can be observed? The eight demos separate information in the inputs, decisions under a threshold, explanations, monitoring, and communication. They use NumPy and scikit-learn so the computations remain visible. These simulated groups are teaching constructs, not estimates about real demographic groups.

## Project research-record checkpoint — governance

Update the single project-root `RESEARCH_RECORD.md`. Every final project now needs a governance boundary, even if fairness or lending is not its main topic:

1. **Purpose and owner:** intended use, prohibited use, accountable owner, and review authority.
2. **Population and exposure:** who or what is scored, who can be affected, horizon, and relevant subgroups.
3. **Monitoring:** live metric and denominator, drift trigger, review cadence, and evidence-retention rule.
4. **Explanation and challenge:** reason or evidence supplied, route to challenge, escalation, and human authority.
5. **Fallback:** stop condition, rollback or safe alternative, known limitation, and current legal or policy authority where applicable.

Link the record to the model card or equivalent artifact. Treat a disparity or drift metric as an alarm to investigate, not a self-interpreting verdict.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)

_demo_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
_figure_dir = _demo_root.parent / "figures" if _demo_root.name == "lab" and _demo_root.parent.name == "week14" else None

def save_teaching_figure(fig, name):
    """Keep local debrief figures identical to notebook output; no Colab writes."""
    if _figure_dir is None or not _figure_dir.exists():
        return
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
    target = _figure_dir / name
    if not target.exists() or target.read_bytes() != buffer.getvalue():
        target.write_bytes(buffer.getvalue())

### Demo 1 — Build a synthetic loan book and expose the proxy

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 1. Can the remaining inputs reveal the removed group?

**Purpose:** show why removing one field does not remove all information about it. `X` has 4,000 applicant rows and four columns: income, debt-to-income ratio (DTI), credit score, and prior delinquencies. DTI divides debt payments by income. `default` and `group` are separate vectors of zeros and ones.

`LogisticRegression(...).fit(X, default)` estimates a default-probability model. `fit(X, group)` estimates a second model with a different target. In `predict_proba(X)[:, 1]`, the colon keeps every applicant and column 1 is the estimated probability of class one. AUC measures ranking: how often a randomly chosen class-one case scores above a class-zero case, counting ties halfway. Chance ranking is 0.5.

The figure compares chance with the audit model's 0.879 AUC. Both models are fitted and scored on the same rows for this mechanism exercise. This is not an independent audit. The chapter figure instead uses 12,000 training rows and a separate 10,000-row audit; do not expect identical results.

**Predict, change, explain:** make the two groups' income and DTI distributions identical, rerun the dependent cells, and predict what happens to group reconstruction. Explain why successful reconstruction alone does not establish unlawful decision-making. A deployment claim needs a frozen model and genuinely separate applicants.

In [ ]:
n = 4000
group = (rng.random(n) < 0.4).astype(int)  # group 1 = minority (~40%)

# The controlled groups have different income and DTI distributions. Group is
# never supplied to the default model, but these differences can travel through
# the allowed features.
income = np.where(group == 1, rng.normal(50, 15, n), rng.normal(70, 20, n))
income = np.clip(income, 10, None)
dti = np.where(group == 1, rng.normal(0.42, 0.10, n), rng.normal(0.29, 0.10, n))
dti = np.clip(dti, 0.05, 0.9)
score = rng.normal(680, 60, n) - 100 * (dti - 0.3)
score = np.clip(score, 350, 850)
delinq = rng.poisson(0.4 + 0.3 * (dti > 0.4), n)

# The simulator uses these financial inputs. This assumption does not establish
# their legal admissibility or fairness in a real decision system.
logit = -2.0 + 5 * (dti - 0.35) + 0.02 * (650 - score) + 0.3 * delinq
p_default = 1 / (1 + np.exp(-logit))
default = (rng.random(n) < p_default).astype(int)

X = np.column_stack([income, dti, score, delinq])
features = ["income", "dti", "score", "delinq"]

# Train a default model that does NOT see group
model = LogisticRegression(max_iter=1000).fit(X, default)

# Now show the proxy: predict GROUP from the "allowed" features
proxy_model = LogisticRegression(max_iter=1000).fit(X, group)
proxy_auc = roc_auc_score(group, proxy_model.predict_proba(X)[:, 1])
print(f"AUC of group reconstructed from allowed features: {proxy_auc:.3f}")
print("In-sample diagnostic: this illustrates recoverable group information, not held-out accuracy.")
# This is a teaching diagnostic on training rows, not an independent audit.
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(["Chance ranking", "Group predicted from financial inputs"], [0.5, proxy_auc],
        color=["gray", "#1565c0"])
ax.set(xlim=(0, 1), xlabel="In-sample group-prediction AUC",
       title="Removing a field can leave its information behind")
ax.text(proxy_auc + 0.015, 1, f"{proxy_auc:.3f}", va="center")
fig.tight_layout()
save_teaching_figure(fig, "demo1_proxy.png")
plt.show()

### Demo 2 — 4/5ths rule and impact ratio

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 2. Count people before dividing rates

**Purpose:** translate the model score into an approval and count who receives it. `predict_proba(X)[:, 0]` is estimated repayment probability, so approval uses `score >= threshold`. If we used default probability instead, approval would use the opposite inequality.

`np.quantile(scores, 0.30)` sets a threshold on these applicants to approve roughly the top 70%. The mask `group == 0` selects Group A's denominator before the mean counts its approvals. The smaller-to-larger rate ratio is `min(rate_A, rate_B) / max(...)`; it is undefined if both rates are zero. The two nonempty groups here have positive approval rates.

The plot displays 1,848/2,394 approvals for A and 952/1,606 for B: 77.2%, 59.3%, and ratio 0.768. Wilson intervals are approximate binomial intervals for a fixed policy; the displayed intervals omit uncertainty from fitting and threshold selection on these same people. The 0.80 line is a screening reference inherited from employment practice, not a lending-law boundary.

**Predict, change, explain:** raise the common repayment threshold and recompute counts, rates, and ratio. Can both groups lose access while their ratio improves? Explain why the simulator's fully observed repayment labels would be unavailable for historical denials in a real loan book.

In [ ]:
# Score each applicant; threshold to approve ~70% overall.
scores = model.predict_proba(X)[:, 0]  # P(not default)
threshold = np.quantile(scores, 0.30)  # approve top 70%
approve = (scores >= threshold).astype(int)

rate_A = approve[group == 0].mean()
rate_B = approve[group == 1].mean()
impact_ratio = min(rate_A, rate_B) / max(rate_A, rate_B)
print(f"Group A approval rate: {rate_A:.3f}")
print(f"Group B approval rate: {rate_B:.3f}")
print(f"Impact ratio:           {impact_ratio:.3f}   "
      f"{'above screen reference' if impact_ratio >= 0.80 else 'SCREEN FLAG'}")

# Wilson intervals describe binomial sampling uncertainty for a fixed policy.
# They do not account for choosing this threshold on these same applicants.
def wilson_interval(successes, count, z=1.96):
    rate = successes / count
    denominator = 1 + z*z/count
    center = (rate + z*z/(2*count)) / denominator
    half = z * np.sqrt(rate*(1-rate)/count + z*z/(4*count*count)) / denominator
    return center - half, center + half

print("Group  applicants  approvals  rate   illustrative 95% interval")
for value, label in [(0, "A"), (1, "B")]:
    count = int(np.sum(group == value))
    successes = int(approve[group == value].sum())
    low, high = wilson_interval(successes, count)
    print(f"  {label}    {count:6d}     {successes:6d}   {successes/count:.3f}  [{low:.3f}, {high:.3f}]")
print("These descriptive intervals omit uncertainty from model and threshold selection.")
group_counts = np.array([(group == g).sum() for g in (0, 1)])
approval_counts = np.array([approve[group == g].sum() for g in (0, 1)])
group_rates = approval_counts / group_counts
intervals = np.array([wilson_interval(k, n) for k, n in zip(approval_counts, group_counts)])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(["Group A", "Group B"], group_rates, color=["#1565c0", "#ef6c00"],
       yerr=np.stack([group_rates - intervals[:, 0], intervals[:, 1] - group_rates]), capsize=5)
for i, (k, count) in enumerate(zip(approval_counts, group_counts)):
    ax.text(i, group_rates[i] + 0.04, f"{k:,} / {count:,} = {group_rates[i]:.1%}", ha="center")
ax.axhline(0.8 * group_rates.max(), color="gray", ls="--", label="80% of higher rate")
ax.set(ylim=(0, 1), ylabel="Approval fraction", title=f"Same applicants, rate ratio = {impact_ratio:.3f}")
ax.legend(fontsize=8)
fig.tight_layout()
save_teaching_figure(fig, "demo2_selection.png")
plt.show()

### Mini-project: audit the decision path

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demos 1–2 — Expose the proxy and calculate the impact ratio**.

Run the proxy and selection-rate cells in `week14_demos.ipynb`.

Produce one page containing:

1. group counts, rates, intervals, and ratio;
2. the proxy AUC;
3. the stage at which outcome labels become selective;
4. one legal or institutional authority that still needs review.

**Research record:** group counts, rates, intervals, and ratio; the proxy AUC; the stage at which outcome labels become selective; the legal or institutional authority still needing review.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 3 — Fairness-policy tradeoff map

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 3. One policy knob changes several outcomes

**Purpose:** compare consequences of different rules before choosing one. Group A's threshold stays at its 20th percentile. The loop changes Group B's quantile from 0.05 to 0.50 across 30 policies. `np.where` applies the appropriate threshold to each row. This group-specific policy family is a mathematical illustration; it is not a recommendation or authorization for real credit decisions.

Each row of `results` contains the smaller/larger approval-rate ratio, loss, total approval fraction, A's rate, and B's rate. The loss is four units for a false approval (approve a default) plus one for a false denial (deny a repayment), divided by all applicants. Those weights express an explicit preference, not a universal economic cost.

The scatterplot puts ratio horizontally, loss vertically, and approval fraction in color. It is a tradeoff map containing all candidate policies, including dominated choices—not an already filtered Pareto frontier. The ratio folds when B's rate passes A's: similar ratios can describe opposite group orderings and different volumes. All rows reuse the training sample; the chapter's separate common-threshold sweep selects on validation and confirms frozen choices on audit data.

**Predict, change, explain:** choose two rows with similar ratios and compare their group rates, loss, and approval volume. Then change the false-approval cost and identify which policy you would examine next. The cell has no review queue, capacity model, or legal test; name those missing judgments rather than claiming the computation resolves them.

In [ ]:
# Hold Group A threshold fixed; sweep Group B threshold.
# Trace (impact ratio, declared decision loss, approval volume).

t_A = np.quantile(scores[group == 0], 0.20)  # ~80% A approval

results = []
for q_B in np.linspace(0.05, 0.50, 30):
    t_B = np.quantile(scores[group == 1], q_B)
    appr = np.where(group == 0, scores >= t_A, scores >= t_B).astype(int)
    rA = appr[group == 0].mean()
    rB = appr[group == 1].mean()
    ir = min(rA, rB) / max(rA, rB)
    false_approval = (appr == 1) & (default == 1)
    false_denial = (appr == 0) & (default == 0)
    loss = (4 * false_approval + false_denial).mean()
    results.append((ir, loss, appr.mean(), rA, rB))

results = np.array(results)
print("IR, decision loss, approval volume, and group rates across the tradeoff map:")
for ir, loss, volume, rA, rB in results[::5]:
    print(f"  IR = {ir:.3f}   loss = {loss:.3f}   approvals = {volume:.3f}   A = {rA:.3f}   B = {rB:.3f}")
# The full sweep is a tradeoff map. Only nondominated points are a Pareto frontier.
fig, ax = plt.subplots(figsize=(7, 3.6))
points = ax.scatter(results[:, 0], results[:, 1], c=results[:, 2], cmap="viridis")
fig.colorbar(points, ax=ax, label="Overall approval fraction")
ax.set(xlabel="Smaller / larger group approval rate", ylabel="Declared decision loss",
       title="Threshold choices change several outcomes together")
fig.tight_layout()
save_teaching_figure(fig, "demo3_policy_tradeoffs.png")
plt.show()

### Mini-project: move along the tradeoff map

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 3 — Fairness-policy tradeoff map**.

Run the threshold sweep in `week14_demos.ipynb`.

Choose two points with different approval volume. For each, report predictive
loss, group rates, and the institutional judgment required to prefer it.

**Research record:** the two approval volumes chosen; predictive loss and group rates at each; the institutional judgment required to prefer one over the other.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — Candidate adverse contributions from the logistic model

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 4. Explain a score, then inspect the actual decision

**Purpose:** distinguish an additive score decomposition from the reason for an implemented denial. A logistic model forms log-odds `intercept + X @ coefficient`; its probability is the sigmoid of that number. Contributions are `coefficient[j] * (applicant[j] - training_mean[j])`, not probability changes.

The cell chooses the first denied applicant, sorts contributions, and prints only positive contributions as adverse relative to this reference. The baseline log-odds plus all contributions exactly reconstruct the applicant's log-odds. Negative contributions lower default risk and must not be called adverse simply because they appear among the two largest values.

The repayment score is approximately 0.833 versus the 0.835 approval threshold. Credit score is the largest adverse contribution; a tiny income contribution is not automatically a principal reason. This cell has no extra eligibility rule or manual override. In a real system, inspect those records and validate reason logic against the applicable requirements; this list is not a compliant notice generator.

**Predict, change, explain:** replace the reference mean with the mean among approved applicants. Check that the corresponding baseline plus contributions still reconstructs the same score. Explain why the decomposition can change although the decision does not.

In [ ]:
# Pick a denied applicant; decompose into per-feature contributions.

# Relative to the training mean, each contribution is beta_j * (x_j - mean_j).
# Contributions add to a change in log-odds, not a change in probability.
mu = X.mean(0)
contribs_all = (X - mu) * model.coef_.ravel()

denied_idx = np.where(approve == 0)[0][0]
print(f"\nApplicant #{denied_idx} (denied):")
print(f"  income={X[denied_idx, 0]:.1f}, dti={X[denied_idx, 1]:.2f}, "
      f"score={X[denied_idx, 2]:.0f}, delinq={X[denied_idx, 3]:.0f}")
contribs = contribs_all[denied_idx]
order = np.argsort(-contribs)  # largest contributions raise predicted default risk
baseline_log_odds = float(model.intercept_[0] + mu @ model.coef_.ravel())
applicant_log_odds = float(model.decision_function(X[denied_idx:denied_idx + 1])[0])
np.testing.assert_allclose(baseline_log_odds + contribs.sum(), applicant_log_odds)
print(f"  baseline log-odds {baseline_log_odds:+.3f} + contributions {contribs.sum():+.3f} = {applicant_log_odds:+.3f}")
print(f"  repayment score {scores[denied_idx]:.3f} < approval threshold {threshold:.3f}")
adverse_order = [j for j in order if contribs[j] > 0]
print("Largest positive contributions relative to the training mean:")
for j in adverse_order[:2]:
    print(f"  {features[j]:8s}: contribution = {contribs[j]:+.3f}")
print("These are candidate model attributions. Reconcile them with the "
      "threshold, policy rules, and overrides before stating decision reasons.")

### Demo 5 — Permutation importance (model-agnostic, pure NumPy)

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 5. Shuffle a feature to test predictive reliance

**Purpose:** measure how much a frozen model's ranking depends on an input. Copy `X`, shuffle one column, predict without refitting, and subtract the shuffled AUC from the original AUC. Copying prevents one shuffle from contaminating the next feature's experiment. A dedicated random generator makes these shuffles independent of other demos.

Here the credit-score shuffle produces the largest AUC drop, about 0.255. Income's drop is near zero. This is a single shuffle per feature on training rows, so it illustrates the calculation rather than estimating stable held-out importance. Correlated features can substitute for each other, and shuffling can create unrealistic combinations. A small drop does not prove a feature has no effect or that the system is fair.

**Predict, change, explain:** repeat several shuffles on a reserved evaluation population and report their distribution. Contrast this global ranking diagnostic with Demo 4's decomposition for one person. Neither establishes a causal effect.

In [ ]:
# Shuffle each column, measure AUC drop. This is what permutation importance
# *is*; the sklearn version is identical in spirit.

rng_permutation = np.random.default_rng(1405)
base_auc = roc_auc_score(default, model.predict_proba(X)[:, 1])
print(f"\nIn-sample base AUC: {base_auc:.4f}")
for j, name in enumerate(features):
    Xp = X.copy()
    rng_permutation.shuffle(Xp[:, j])
    auc_p = roc_auc_score(default, model.predict_proba(Xp)[:, 1])
    print(f"  shuffle {name:8s}: AUC = {auc_p:.4f}   drop = {base_auc - auc_p:+.4f}")
# A small drop can also reflect redundant information or an unsuitable metric.
# This cell uses training rows to explain the mechanics. For model assessment,
# repeat on a reserved evaluation sample; do not call these drops held-out damage.

### Demo 6 — Drift monitoring (PSI + delayed outcomes) over 24 months

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 6. Watch inputs now and outcomes when they arrive

**Purpose:** make the two monitoring clocks executable. The frozen default model scores 24 synthetic monthly cohorts. Income drifts downward, but income does not enter the planted default equation. At cohort 13, the true log-odds intercept rises by 0.8, increasing default risk without necessarily destroying ranking. The cohort rows reuse the original financial profiles; this is a controlled change experiment, not 24 independent loan books.

`psi` uses baseline quantiles as fixed bins, with infinite outside edges so shifted observations remain counted. Smoothed counts are normalized to proportions; the sum of `(current - baseline) * log(current / baseline)` is Population Stability Index. Zero means identical binned distributions. The 0.25 line is an illustrative convention, not a calibrated statistical alarm or regulatory limit.

For each cohort, the code calculates AUC and `observed_default_rate - mean_prediction`. The latter checks average risk calibration only: positive means average default risk is understated; zero can conceal opposing errors across scores or groups.

`available_months = cohort_months + 3` moves outcome evidence to the date it can be used. The dashboard shows input PSI for the current month and outcomes only for mature cohorts. The first changed cohort is 13, so its results first appear in calendar month 16. The final three cohorts have no mature labels by month 24. The simulator knows those outcomes internally; the dashboard deliberately withholds them.

**Predict, change, explain:** set the label lag to six months. Predict the first calendar month with changed outcome evidence before running it. Then remove the intercept change while retaining the income drift. Explain why an input alert and a performance failure are different findings. The chapter's monitor uses a slope change, 30 months, and an empirically calibrated PSI limit; its numbers and alert date differ deliberately.

In [ ]:
def psi(base, cur, bins=10):
    edges = np.quantile(base, np.linspace(0, 1, bins + 1))
    # Freeze baseline quantile cuts, but retain observations beyond its range.
    # A shifted population must not disappear from the histogram denominator.
    edges = np.unique(edges)
    if len(edges) < 2:
        raise ValueError("PSI needs a nonconstant baseline")
    edges[0], edges[-1] = -np.inf, np.inf
    p_base, _ = np.histogram(base, bins=edges)
    p_cur, _ = np.histogram(cur, bins=edges)
    if p_cur.sum() == 0:
        raise ValueError("PSI needs finite current observations")
    # Smooth counts, then normalize so each distribution sums to one.
    p_base = (p_base + 1e-6) / (p_base.sum() + 1e-6 * len(p_base))
    p_cur = (p_cur + 1e-6) / (p_cur.sum() + 1e-6 * len(p_cur))
    return float(np.sum((p_cur - p_base) * np.log(p_cur / p_base)))

# Freeze the model. Income shifts without entering the true default equation.
# At cohort month 13, only the true intercept jumps: risk rises at the same inputs.
# Predictions and input diagnostics exist now; labels become usable three months later.
rng_monitor = np.random.default_rng(1406)
base_income = X[:, 0].copy()
months = np.arange(1, 25)
maturity_lag = 3
change_cohort = 13
aucs, psis, observed_rates, predicted_rates = [], [], [], []
for month in months:
    income_m = base_income - 0.5 * month + rng_monitor.normal(0, 3, n)
    X_m = X.copy(); X_m[:, 0] = income_m
    relationship_shift = 0.8 if month >= change_cohort else 0.0
    logit_m = (-2.0 + 5 * (X_m[:, 1] - 0.35) + 0.02 * (650 - X_m[:, 2])
               + 0.3 * X_m[:, 3] + relationship_shift)
    p = 1 / (1 + np.exp(-logit_m))
    y_m = (rng_monitor.random(n) < p).astype(int)
    prediction = model.predict_proba(X_m)[:, 1]
    aucs.append(roc_auc_score(y_m, prediction))
    psis.append(psi(base_income, income_m))
    observed_rates.append(y_m.mean())
    predicted_rates.append(prediction.mean())
aucs, psis = np.array(aucs), np.array(psis)
observed_rates, predicted_rates = np.array(observed_rates), np.array(predicted_rates)
calibration_gap = observed_rates - predicted_rates  # one aggregate check, not full calibration
available_months = months + maturity_lag
visible = available_months <= months[-1]
print("\nCalendar month  input PSI  outcome cohort  matured AUC  observed-minus-predicted")
for month in months:
    cohort = month - maturity_lag
    if cohort < 1:
        print(f"{month:5d}          {psis[month-1]:.3f}       --             --           --")
    else:
        k = cohort - 1
        print(f"{month:5d}          {psis[month-1]:.3f}       {cohort:2d}            {aucs[k]:.3f}        {calibration_gap[k]:+.3f}")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
axes[0].plot(months, psis, marker=".")
axes[0].axhline(0.25, ls="--", color="gray", label="Illustrative review trigger")
axes[0].set(ylabel="PSI of income", title="Inputs are visible now")
axes[0].legend(fontsize=7)
axes[1].plot(available_months[visible], aucs[visible], marker=".", color="#ef6c00")
axes[1].set(ylabel="AUC on matured cohort", ylim=(0.5, 1), title="Ranking can remain useful")
axes[2].plot(available_months[visible], calibration_gap[visible], marker=".", color="#a32121")
axes[2].axhline(0, color="gray", ls="--")
axes[2].set(ylabel="Observed minus predicted default rate", title="Average risk can be understated")
for ax in axes:
    ax.set(xlabel="Calendar month", xlim=(1, 24))
    ax.axvline(change_cohort, color="gray", ls=":", label="Planted change")
for ax in axes[1:]:
    ax.axvline(change_cohort + maturity_lag, color="#a32121", ls="--")
fig.suptitle(f"Change in cohort {change_cohort} → first changed labels visible in month {change_cohort + maturity_lag}", fontsize=11)
fig.tight_layout()
save_teaching_figure(fig, "demo6_monitoring.png")
plt.show()
print(f"The final {maturity_lag} cohorts have no mature outcome evidence in this 24-month view.")
print("A nonzero mean calibration gap diagnoses average risk mismatch; zero does not prove full calibration.")

### Demo 7 — Model card stub (what you'd produce alongside the model)

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 7. Turn the results into a record of evidence and gaps

**Purpose:** summarize what has actually been tested and what remains unknown. The formatted string inserts the earlier AUC, counts-derived rates, ratio, and proxy AUC. It labels them in-sample. It lists missing held-out evaluation, subgroup outcomes, reason validation, operational testing, and current legal review instead of declaring the model ready to use.

An owner, response deadline, tested fallback, authority, and capacity are explicit fields to complete. An alert starts an investigation; automatic retraining or automatic manual referral is not assumed to be a safe remedy.

**Try and explain:** replace the placeholders with a proposed classroom governance record, marking untested statements as such. Use its evidence-gap list to storyboard and critique the project. Keep the separate `RESEARCH_RECORD.md` linking the information clock, frozen plan, trials, final evidence, limitations, and artifacts. A model card is a summary, not a substitute for the research record or a production approval.

In [ ]:
card = """
MODEL CARD — Default Risk Model v1.0
====================================
Intended use:    Approve/deny consumer credit; threshold tuned for ~70% approval.
Training data:   4,000 synthetic applicants; classroom illustration only.
                 Real use needs a new approved data, design, and validation process.
Features:        income, dti, credit score, prior delinquencies.
                 Audit-group field is EXCLUDED from the default-model inputs.

Performance:
    In-sample AUC:                {auc:.3f}
    Group A approval rate:        {rA:.3f}
    Group B approval rate:        {rB:.3f}
    Impact ratio (4/5ths rule):   {ir:.3f}  ({status})

Known limitations:
    - Group information remains in the inputs (in-sample reconstruction AUC = {pa:.3f}).
      This diagnostic alone does not establish unlawful discrimination.
    - Thresholds and diagnostics above reuse the training sample; evaluate
      a frozen policy on reserved applicants before making performance claims.
    - No monitoring deployed yet; this demo investigates PSI above 0.25
      (a convention, not a regulatory or statistical threshold).

Governance:
    - Owner: <named human>
    - Review: model-risk committee, quarterly
    - Alert owner and response time: <name and deadline>
    - Response: investigate data, exposure, and mature outcomes before changing models.
    - Fallback: <tested policy, authority, capacity, and activation conditions>
    - Missing evidence: held-out loss/calibration, subgroup outcomes, selective-label
      analysis, validated reason logic, legal review, and operational testing.
    - Research record: link the data clock, frozen plan, trial history, final result,
      limitations, and artifact index; this card does not replace that record.
""".format(
    auc=base_auc, rA=rate_A, rB=rate_B, ir=impact_ratio,
    status="SCREEN FLAG" if impact_ratio < 0.80 else "above screen reference",
    pa=proxy_auc,
)
print(card)

### Mini-project: storyboard the evidence chain

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 7 — Model card stub**, then turn its evidence fields into the storyboard below.

Lay out 12–16 blank boxes.

Write one sentence in each box stating what that slide proves. Remove any slide
whose sentence does not advance the argument. Mark the one visual each remaining
slide needs.

**Research record:** the one-sentence claim for each box; the slides you removed and why; the visual each surviving slide still needs.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 8 — Repair a result slide without changing the result

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 8. Repair the evidence without changing the result

**Purpose:** show how a result slide becomes more auditable without changing its underlying numbers. Before the code runs, the supplied weak slide deliberately hides the population, baseline, units, costs, and evaluation period behind anonymous bars; it does **not** reveal those omissions for the student. Demo 8 then reveals the diagnosis in its left panel. The right panel uses the same illustrative gross values, adds net-of-cost values, names the transparent baseline and machine-learning policy, and states the period and repeated-test context.

The arrays `gross` and `net` are deterministic classroom values, not market evidence. The chart is explicitly labeled **Synthetic teaching example**. The repair does not prove that the test was well designed; it exposes the claims that would need to be checked in the research record. It also preserves one visible limitation: the distribution across the six seeded tests is still missing.

**Predict, change, explain:** before running the cell, list the evidence missing from the weak panel and classify each gap. A **presentation failure** means the evidence exists but the slide hides it; an **evidence failure** means the study never produced it. Then change the assumed transaction cost and update only the net values and conclusion. Explain why a visual repair can narrow or reverse the headline without creating a new observation—or a missing uncertainty distribution.

In [ ]:
gross = np.array([1.82, 1.70])
net = np.array([1.65, 1.42])

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.25))
ax = axes[0]
ax.bar(["A", "B"], gross, color=["#6b7280", "#1565c0"])
ax.set(ylim=(0, 2.1), title="Results", ylabel="performance")
ax.text(.5, .90, "Missing: population • baseline • costs • test dates",
        transform=ax.transAxes, ha="center", color="#a32121", fontsize=8.5,
        fontweight="bold", bbox=dict(facecolor="white", edgecolor="#a32121",
                                      boxstyle="round,pad=.3"))
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
x = np.arange(2); width = .34
ax.bar(x-width/2, gross, width, color="#b9c2cc", label="gross")
ax.bar(x+width/2, net, width, color=["#2e7d32", "#a32121"],
       label="net of 50 bp cost")
ax.set(xticks=x, xticklabels=["transparent baseline", "ML policy"],
       ylim=(0, 2.1), ylabel="wealth multiple",
       title="Synthetic teaching example:\nML policy trails the baseline after costs")
for i, value in enumerate(net):
    ax.text(i+width/2, value+.05, f"{value:.2f}×", ha="center",
            fontsize=9, fontweight="bold")
ax.legend(fontsize=8)
ax.set_xlabel("Illustrative 2016–2026 test • same mandate • 50 bp cost",
              fontsize=8.5, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("A repair changes what the audience can verify—not the underlying result",
             color="#18376b", fontsize=13, fontweight="bold")
fig.text(.73, .015, "Still missing: uncertainty across the six seeded tests",
         ha="center", color="#a32121", fontsize=8.8, fontweight="bold")
fig.tight_layout(rect=[0, .055, 1, 1])
save_teaching_figure(fig, "demo8_slide_repair.png")
plt.show()

print("Illustrative net wealth multiples: baseline = 1.65x, ML policy = 1.42x.")
print("The repair exposes the baseline, period, cost, and repeated-test context; it does not create new evidence.")

### Mini-project: repair the weakest slide

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Start with the supplied weak slide. Then audit the project slide you trust least. A partner is optional.

Answer:

1. What claim does this slide support?
2. What comparison is missing?
3. What can become a picture, plot, table, or diagram?
4. What sentence should remain after the repair?
5. Is each gap a **presentation failure** or an **evidence failure**?

**Research record:** claim; missing comparison; presentation/evidence gaps; visual change; surviving sentence.

**Notebook:** write your diagnosis, run **Demo 8**, and compare repairs.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

## Where this leaves us

Responsible deployment requires more than a fitted score: it requires population-specific evidence, a declared policy, understandable reasons, drift monitoring, escalation, and a record of who owns each decision. No single metric settles fairness or model risk.

Week 15 brings the course together by asking each project to survive costs, search accounting, time-aware evaluation, regime checks, and an explicit red team.